# KV Cache — Fast Autoregressive Inference

## The Problem with Naive Inference

LLM generation is autoregressive — generate one token, append it, generate the next.

**Naive approach:** for every new token, recompute K and V for ALL previous tokens.

```
Step 1: tokens=[1]           → compute K,V for 1 token  → generate token 2
Step 2: tokens=[1,2]         → compute K,V for 2 tokens → generate token 3
Step 3: tokens=[1,2,3]       → compute K,V for 3 tokens → generate token 4
...
Step 100: tokens=[1..100]    → compute K,V for 100 tokens → generate token 101

Total KV computations: 1 + 2 + 3 + ... + 100 = O(N²) — quadratic!
```

**KV Cache:** compute K and V once per token and **cache** them.
```
Step 1: compute K,V for token 1 → cache it → generate token 2
Step 2: compute K,V for token 2 only → append to cache → generate token 3
...
Total KV computations: 1 + 1 + ... + 1 = O(N) — linear!
```

## Resources

| Resource | Link |
|---|---|
| Illustrated KV cache (dipkumar.dev) | https://www.dipkumar.dev/becoming-the-unbeatable/posts/gpt-kvcache/ |
| PagedAttention / vLLM (efficient KV cache mgmt) | https://arxiv.org/abs/2309.06180 |
| vLLM blog — continuous batching | https://blog.vllm.ai/2023/06/20/vllm.html |
| HuggingFace — how KV cache works | https://huggingface.co/docs/transformers/kv_cache |
| Quantizing the KV cache (TurboQuant preview) | see 07-turboquant.ipynb |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


class CachedAttention(nn.Module):
    """Attention with KV caching for inference."""

    def __init__(self, d_model, n_heads, max_seq_len=512):
        super().__init__()
        self.n_heads = n_heads
        self.d_k     = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

        # KV cache — pre-allocated for max sequence length
        # In production: [batch, n_heads, max_seq_len, d_k]
        self.cache_k   = None
        self.cache_v   = None
        self.cache_len = 0

    def reset_cache(self):
        self.cache_k   = None
        self.cache_v   = None
        self.cache_len = 0

    def forward(self, x, use_cache=False):
        """
        x: [batch, seq_len, d_model]
           During training: seq_len = full sequence
           During inference with cache: seq_len = 1 (just the new token)
        """
        batch, seq_len, _ = x.shape

        Q = self.W_q(x).reshape(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).reshape(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).reshape(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)

        if use_cache:
            # Append new K, V to cache
            if self.cache_k is None:
                self.cache_k = K
                self.cache_v = V
            else:
                self.cache_k = torch.cat([self.cache_k, K], dim=2)  # grow along seq dim
                self.cache_v = torch.cat([self.cache_v, V], dim=2)

            # Attend over ALL cached K, V but only current Q
            K = self.cache_k
            V = self.cache_v

        scores  = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        weights = F.softmax(scores, dim=-1)
        out     = (weights @ V).transpose(1, 2).reshape(batch, seq_len, -1)
        return self.W_o(out)

In [ ]:
class MiniGPTWithCache(nn.Module):
    """Minimal autoregressive model with KV cache support."""

    def __init__(self, vocab_size=100, d_model=64, n_heads=4, n_layers=2, max_len=128):
        super().__init__()
        self.embed  = nn.Embedding(vocab_size, d_model)
        self.pos    = nn.Embedding(max_len, d_model)
        self.layers = nn.ModuleList([CachedAttention(d_model, n_heads) for _ in range(n_layers)])
        self.norm   = nn.LayerNorm(d_model)
        self.head   = nn.Linear(d_model, vocab_size, bias=False)
        self.pos_counter = 0

    def reset_cache(self):
        for layer in self.layers:
            layer.reset_cache()
        self.pos_counter = 0

    def forward(self, token_ids, use_cache=False):
        batch, seq_len = token_ids.shape
        positions = torch.arange(self.pos_counter, self.pos_counter + seq_len, device=token_ids.device)

        x = self.embed(token_ids) + self.pos(positions)

        for layer in self.layers:
            x = x + layer(self.norm(x), use_cache=use_cache)

        if use_cache:
            self.pos_counter += seq_len

        return self.head(self.norm(x))


# --- Autoregressive generation with KV cache ---
@torch.no_grad()
def generate(model, prompt_ids, max_new_tokens=20):
    model.eval()
    model.reset_cache()

    # Prefill — process the prompt in one shot
    tokens     = prompt_ids
    logits     = model(tokens, use_cache=True)   # [1, prompt_len, vocab]
    next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)  # [1, 1]
    tokens     = torch.cat([tokens, next_token], dim=1)

    # Decode — generate one token at a time using cache
    for step in range(max_new_tokens - 1):
        logits     = model(next_token, use_cache=True)   # [1, 1, vocab] — only 1 token!
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        tokens     = torch.cat([tokens, next_token], dim=1)

    return tokens


model  = MiniGPTWithCache()
prompt = torch.randint(0, 100, (1, 5))   # batch=1, prompt=5 tokens
out    = generate(model, prompt, max_new_tokens=10)
print('Prompt:   ', prompt[0].tolist())
print('Generated:', out[0].tolist())

In [ ]:
# Speed comparison — naive vs cached generation
import time

@torch.no_grad()
def generate_naive(model, prompt_ids, max_new_tokens=20):
    """No KV cache — recomputes K,V for all tokens at every step."""
    model.eval()
    tokens = prompt_ids
    for _ in range(max_new_tokens):
        logits     = model(tokens, use_cache=False)  # full sequence every time
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        tokens     = torch.cat([tokens, next_token], dim=1)
    return tokens


model  = MiniGPTWithCache(vocab_size=1000, d_model=128, n_heads=4, n_layers=4)
prompt = torch.randint(0, 1000, (1, 32))
N      = 50  # tokens to generate

t0 = time.time()
for _ in range(3):
    generate_naive(model, prompt, max_new_tokens=N)
naive_ms = (time.time() - t0) / 3 * 1000

t0 = time.time()
for _ in range(3):
    generate(model, prompt, max_new_tokens=N)
cached_ms = (time.time() - t0) / 3 * 1000

print(f'Naive (no cache):  {naive_ms:.1f} ms')
print(f'With KV cache:     {cached_ms:.1f} ms')
print(f'Speedup:           {naive_ms/cached_ms:.1f}x')